**Cell 1: Import Libraries and Load Data**  
This cell imports the necessary Python libraries for data manipulation, statistical testing, and hypothesis testing. It then loads the billings features dataset from the processed data directory.  
- **Purpose**: Set up the environment and load the dataset for analysis.  
- **Key Libraries**:  
  - `pandas` and `numpy`: For data handling and numerical operations.  
  - `scipy.stats`: For statistical tests like t-test and Mann-Whitney U test.  
  - `statsmodels.stats.multitest`: For correcting p-values in multiple testing scenarios.  
- **No hypothesis testing in this cell** (preparatory step).  
- **Expected Output**: A preview of the first few rows of the billings dataset.

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import multipletests

df = pd.read_csv("../../data/03_final/final_billings_features.csv")

df.head()

,co_ref,total_spent,avg_payment,num_payments,last_payment_date,cutoff_date,days_since_last_payment,payments_last_30,payments_last_90,spend_last_30,spend_last_90,payment_trend,tenure_years,payment_method_mode,prospect_outcome
0,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,0.0,0.0,4.0,CARD,Won
1,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,0.0,0.0,3.0,CARD,Won
2,AA0794,2927,975.666667,3,2025-01-10,2025-06-26,167,0.0,0.0,0.0,0.0,0.0,2.0,CARD,Won
3,AA0794,2927,975.666667,3,2025-01-10,2024-06-26,-198,0.0,0.0,0.0,0.0,0.0,4.0,CARD,Won
4,AA0794,2927,975.666667,3,2025-01-10,2024-06-26,-198,0.0,0.0,0.0,0.0,0.0,3.0,CARD,Won


**Cell 2: Create Target Variable**  
This cell creates a binary target variable based on the prospect outcome, where 1 indicates churn ("Churned") and 0 indicates retention. It then displays the distribution of the target variable.  
- **Purpose**: Define the dependent variable for hypothesis testing (churn vs. non-churn).  
- **Key Operation**: Converts the categorical "prospect_outcome" column into a numerical binary variable.  
- **No formal hypothesis testing in this cell** (data preparation).  
- **Expected Output**: Value counts showing the number of churned (1) and non-churned (0) customers.

**Cell 1: Import Libraries and Load Data**  
This cell imports the necessary Python libraries for data manipulation, statistical testing, and hypothesis testing. It then loads the billings features dataset from the processed data directory.  
- **Purpose**: Set up the environment and load the dataset for analysis.  
- **Key Libraries**:  
  - `pandas` and `numpy`: For data handling and numerical operations.  
  - `scipy.stats`: For statistical tests like t-test and Mann-Whitney U test.  
  - `statsmodels.stats.multitest`: For correcting p-values in multiple testing scenarios.  
- **No hypothesis testing in this cell** (preparatory step).  
- **Expected Output**: A preview of the first few rows of the billings dataset.

In [10]:
df["target"] = (df["prospect_outcome"] == "Churned").astype(int)

df["target"].value_counts()

target
0    102310
1     15011
Name: count, dtype: int64

**Cell 3: Define Feature Columns**  
This cell defines the lists of numerical and categorical features from the billings data that will be tested for associations with churn.  
- **Purpose**: Specify which features to include in the hypothesis tests.  
- **Numerical Features**: Continuous or discrete variables (e.g., total_spent, tenure_years).  
- **Categorical Features**: Nominal variables (e.g., payment_method_mode).  
- **No hypothesis testing in this cell** (feature selection).  
- **Expected Output**: None (variable definitions only).

In [11]:
num_cols = [
    "total_spent",
    "avg_payment",
    "num_payments",
    "days_since_last_payment",
    "payments_last_30",
    "payments_last_90",
    "spend_last_30",
    "spend_last_90",
    "payment_trend",
    "tenure_years"
]

cat_cols = [
    "payment_method_mode"
]

**Cell 4: Hypothesis Testing for Numerical Features**  
This cell performs statistical tests on each numerical feature to compare means between churn and non-churn groups. It uses both parametric (t-test) and non-parametric (Mann-Whitney U) tests, along with effect size calculation.  
- **Purpose**: Determine if numerical features differ significantly between churn and non-churn customers.  
- **Null Hypothesis (H₀)**: There is no difference in the mean/median of the feature between churn and non-churn groups (μ_churn = μ_non-churn or median_churn = median_non-churn).  
- **Alternative Hypothesis (H₁)**: There is a significant difference in the mean/median of the feature between churn and non-churn groups (μ_churn ≠ μ_non-churn or median_churn ≠ median_non-churn).  
- **Tests Used**:  
  - **T-test**: Assumes normality; compares means.  
  - **Mann-Whitney U**: Non-parametric; compares medians/distributions.  
  - **Effect Size (Cohen's d)**: Measures the magnitude of the difference (small: 0.2, medium: 0.5, large: 0.8).  
- **Key Insights**: Low p-values (< 0.05) suggest rejection of H₀. Effect size helps interpret practical significance.  
- **Expected Output**: A DataFrame with test results (p-values, means, effect sizes) for each numerical feature.

In [12]:
num_results = []

for col in num_cols:
    churn = df[df["target"] == 1][col].dropna()
    non_churn = df[df["target"] == 0][col].dropna()

    # T-test
    t_stat, t_p = ttest_ind(churn, non_churn, equal_var=False)

    # Mann Whitney
    u_stat, u_p = mannwhitneyu(churn, non_churn, alternative="two-sided")

    # effect size (Cohen's d)
    pooled_std = np.sqrt((churn.var() + non_churn.var()) / 2)
    effect_size = (churn.mean() - non_churn.mean()) / pooled_std if pooled_std != 0 else 0

    num_results.append({
        "feature": col,
        "churn_mean": churn.mean(),
        "non_churn_mean": non_churn.mean(),
        "ttest_p_value": t_p,
        "mannwhitney_p_value": u_p,
        "effect_size": effect_size
    })

num_results = pd.DataFrame(num_results)
num_results

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size
0,total_spent,2289.688095,3678.087978,0.000000e+00,0.000000e+00,-0.749575
1,avg_payment,902.755979,1130.566934,0.000000e+00,0.000000e+00,-0.433856
2,num_payments,2.508227,3.237162,0.000000e+00,0.000000e+00,-0.928659
3,days_since_last_payment,-101.205982,-237.040289,0.000000e+00,0.000000e+00,0.391097
4,payments_last_30,0.196523,0.259242,3.603564e-22,5.896370e-02,-0.077352
5,payments_last_90,0.573646,0.806519,1.644563e-114,2.021271e-32,-0.181061
6,spend_last_30,187.426887,299.019236,2.472632e-54,6.442269e-02,-0.112270
7,spend_last_90,518.698688,916.211993,4.242705e-284,2.754201e-34,-0.254772
8,payment_trend,0.000000,0.000000,NaN,1.000000e+00,0.000000
9,tenure_years,5.299447,7.773424,0.000000e+00,0.000000e+00,-0.501097


**Cell 5: Hypothesis Testing for Categorical Features**  
This cell performs a chi-square test of independence for each categorical feature to check for associations with churn.  
- **Purpose**: Determine if categorical features are associated with churn status.  
- **Null Hypothesis (H₀)**: There is no association between the categorical feature and churn (the feature is independent of churn).  
- **Alternative Hypothesis (H₁)**: There is a significant association between the categorical feature and churn (the feature is dependent on churn).  
- **Test Used**: **Chi-Square Test of Independence** - Compares observed vs. expected frequencies in a contingency table.  
- **Key Insights**: Low p-values (< 0.05) suggest rejection of H₀, indicating the categorical variable influences churn.  
- **Expected Output**: A DataFrame with chi-square p-values for each categorical feature.

In [13]:
cat_results = []

for col in cat_cols:
    cont_table = pd.crosstab(df[col], df["target"])
    chi2, p, dof, expected = chi2_contingency(cont_table)

    cat_results.append({
        "feature": col,
        "chi2_p_value": p
    })

cat_results = pd.DataFrame(cat_results)
cat_results

,feature,chi2_p_value
0,payment_method_mode,0.0


**Cell 6: Apply Multiple Testing Correction**  
This cell applies the Benjamini-Hochberg False Discovery Rate (FDR) correction to the p-values from the numerical feature tests to account for multiple comparisons.  
- **Purpose**: Adjust p-values to control the false positive rate when testing many features simultaneously.  
- **Method**: FDR correction (fdr_bh) - Balances discovery of true positives while limiting false positives.  
- **Key Operation**: Adds "adjusted_p" and "significant" columns (significant if adjusted_p < 0.05).  
- **No new hypotheses** (correction of existing tests).  
- **Expected Output**: The updated DataFrame sorted by adjusted p-values, highlighting significant features.

In [14]:
num_results["adjusted_p"] = multipletests(
    num_results["mannwhitney_p_value"],
    method="fdr_bh"
)[1]

num_results["significant"] = num_results["adjusted_p"] < 0.05

num_results.sort_values("adjusted_p")

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
0,total_spent,2289.688095,3678.087978,0.000000e+00,0.000000e+00,-0.749575,0.000000e+00,True
1,avg_payment,902.755979,1130.566934,0.000000e+00,0.000000e+00,-0.433856,0.000000e+00,True
2,num_payments,2.508227,3.237162,0.000000e+00,0.000000e+00,-0.928659,0.000000e+00,True
3,days_since_last_payment,-101.205982,-237.040289,0.000000e+00,0.000000e+00,0.391097,0.000000e+00,True
9,tenure_years,5.299447,7.773424,0.000000e+00,0.000000e+00,-0.501097,0.000000e+00,True
7,spend_last_90,518.698688,916.211993,4.242705e-284,2.754201e-34,-0.254772,4.590334e-34,True
5,payments_last_90,0.573646,0.806519,1.644563e-114,2.021271e-32,-0.181061,2.887529e-32,True
4,payments_last_30,0.196523,0.259242,3.603564e-22,5.896370e-02,-0.077352,7.158077e-02,False
6,spend_last_30,187.426887,299.019236,2.472632e-54,6.442269e-02,-0.112270,7.158077e-02,False
8,payment_trend,0.000000,0.000000,NaN,1.000000e+00,0.000000,1.000000e+00,False


**Cell 7: Display Significant Features**  
This cell filters and displays only the numerical features that were found to be statistically significant after multiple testing correction.  
- **Purpose**: Summarize the key findings by showing features with strong evidence of association with churn.  
- **Key Insights**: These features are most likely to be useful for churn prediction models.  
- **Expected Output**: A DataFrame of significant features (adjusted_p < 0.05).

In [15]:
significant_features = num_results[num_results["significant"] == True]
significant_features

,feature,churn_mean,non_churn_mean,ttest_p_value,mannwhitney_p_value,effect_size,adjusted_p,significant
0,total_spent,2289.688095,3678.087978,0.000000e+00,0.000000e+00,-0.749575,0.000000e+00,True
1,avg_payment,902.755979,1130.566934,0.000000e+00,0.000000e+00,-0.433856,0.000000e+00,True
2,num_payments,2.508227,3.237162,0.000000e+00,0.000000e+00,-0.928659,0.000000e+00,True
3,days_since_last_payment,-101.205982,-237.040289,0.000000e+00,0.000000e+00,0.391097,0.000000e+00,True
5,payments_last_90,0.573646,0.806519,1.644563e-114,2.021271e-32,-0.181061,2.887529e-32,True
7,spend_last_90,518.698688,916.211993,4.242705e-284,2.754201e-34,-0.254772,4.590334e-34,True
9,tenure_years,5.299447,7.773424,0.000000e+00,0.000000e+00,-0.501097,0.000000e+00,True
